In [0]:
import json
import logging
import re
import uuid
from datetime import datetime

from delta.tables import DeltaTable
from pyspark.sql import Row
from pyspark.sql.functions import (
    current_timestamp, lit, col, sha2, concat_ws, coalesce
)
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, DoubleType, TimestampType
)



#### Logger setup

Sets up a logger so each step prints a timestamped status line during the run.

In [0]:
logger = logging.getLogger("cdr_bronze_ingestion")
logger.setLevel(logging.INFO)

if not logger.handlers:
    handler = logging.StreamHandler()
    formatter = logging.Formatter(
        fmt="%(asctime)s | %(levelname)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )
    handler.setFormatter(formatter)
    logger.addHandler(handler)

logger.info("Logger initialised.")

#### Parameters

Widgets accept input from a person running this notebook directly, or from a job task passing values in.
`run_id` can be shared with the silver notebook so both layers log the same run.

In [0]:
dbutils.widgets.text("base_folder_path", "s3://mimic-iv-datasets/mimic-iv-dataset-3.1")
dbutils.widgets.text("run_id", "")
dbutils.widgets.dropdown("load_type", "auto", ["auto", "historical", "incremental"])
dbutils.widgets.text("table_filter", "")
dbutils.widgets.text("table_config_path", "/Volumes/cdr_bronze/config/table_config/bronze_config.json")

base_folder_path = dbutils.widgets.get("base_folder_path")
run_id_param = dbutils.widgets.get("run_id").strip()
load_type_param = dbutils.widgets.get("load_type").strip().lower()
table_filter_raw = dbutils.widgets.get("table_filter").strip()
table_config_path = dbutils.widgets.get("table_config_path")

run_id = run_id_param if run_id_param else str(uuid.uuid4())

table_filter = (
    {t.strip().lower() for t in table_filter_raw.split(",") if t.strip()}
    if table_filter_raw else None
)

catalog = "cdr_bronze"
schema = "raw_mimic_iv"
audit_table = f"{catalog}.{schema}.ingestion_audit_log"

logger.info(f"run_id            = {run_id}")
logger.info(f"load_type_param   = {load_type_param}")
logger.info(f"base_folder_path  = {base_folder_path}")
logger.info(f"table_filter      = {table_filter or 'ALL TABLES'}")
logger.info(f"table_config_path = {table_config_path}")
logger.info(f"audit_table       = {audit_table}")

#### Load table config

Loads the merge key for each known table from a JSON file in a Unity Catalog volume, so keys can be updated without changing the notebook.
A table set to `null` or missing from the file has no reliable key and falls back to a row hash.

In [0]:
for sql in [
    "CREATE CATALOG IF NOT EXISTS cdr_bronze",
    "CREATE CATALOG IF NOT EXISTS cdr_silver",
    "CREATE SCHEMA  IF NOT EXISTS cdr_bronze.config",
    "CREATE SCHEMA  IF NOT EXISTS cdr_silver.config",
    "CREATE VOLUME  IF NOT EXISTS cdr_bronze.config.table_config",
    "CREATE VOLUME  IF NOT EXISTS cdr_silver.config.silver_config",
]:
    spark.sql(sql)
    print("ok:", sql)

In [0]:
bronze_config = """{
    "patients":           ["subject_id"],
    "admissions":         ["hadm_id"],
    "diagnoses_icd":      ["subject_id", "hadm_id", "seq_num", "icd_code", "icd_version"],
    "procedures_icd":     ["subject_id", "hadm_id", "seq_num", "icd_code", "icd_version"],
    "drgcodes":           ["subject_id", "hadm_id", "drg_type", "drg_code"],
    "services":           ["subject_id", "hadm_id", "transfertime", "prev_service", "curr_service"],
    "transfers":          ["transfer_id"],
    "provider":           ["provider_id"],
    "caregiver":          ["caregiver_id"],
    "poe":                ["poe_id"],
    "poe_detail":         ["poe_id", "field_name", "field_value"],
    "prescriptions":      ["subject_id", "hadm_id", "pharmacy_id", "poe_id", "starttime", "stoptime", "drug_type", "drug"],
    "pharmacy":           ["pharmacy_id"],
    "d_hcpcs":            ["code"],
    "hcpcsevents":        ["subject_id", "hadm_id", "hcpcs_cd", "seq_num"],
    "emar":               ["emar_id"],
    "emar_detail":        null,
    "d_icd_procedures":   ["icd_code", "icd_version"],
    "d_icd_diagnoses":    ["icd_code", "icd_version"],
    "labevents":          ["labevent_id"],
    "icustays":           ["stay_id"],
    "d_items":            ["itemid"],
    "d_labitems":         ["itemid"],
    "procedureevents":    ["subject_id", "hadm_id", "stay_id", "caregiver_id", "itemid", "starttime", "endtime", "storetime", "orderid"],
    "datetimeevents":     ["subject_id", "hadm_id", "stay_id", "caregiver_id", "itemid", "charttime", "storetime"],
    "outputevents":       ["subject_id", "hadm_id", "stay_id", "caregiver_id", "itemid", "charttime", "storetime"],
    "ingredientevents":   ["subject_id", "hadm_id", "stay_id", "caregiver_id", "itemid", "starttime", "endtime", "storetime", "orderid"],
    "inputevents":        ["subject_id", "hadm_id", "stay_id", "caregiver_id", "itemid", "starttime", "endtime", "storetime", "orderid"],
    "omr":                ["subject_id", "chartdate", "seq_num", "result_name"],
    "microbiologyevents": ["microevent_id"],
    "chartevents":        null
}"""

dbutils.fs.put("/Volumes/cdr_bronze/config/table_config/bronze_config.json",
               bronze_config, overwrite=True)

# quick check it parses
import json
print("tables:", len(json.loads(bronze_config)))
print(dbutils.fs.ls("/Volumes/cdr_bronze/config/table_config/"))

In [0]:
with open(table_config_path, "r") as f:
    TABLE_CONFIG = json.load(f)

logger.info(
    f"TABLE_CONFIG loaded for {len(TABLE_CONFIG)} known tables from {table_config_path}. "
    f"Tables discovered but not listed here fall back to a row hash key."
)

#### CSV discovery

Recursively scans the source S3 folder and matches each CSV to a clean bronze table name.

In [0]:

#To find the path to each csv file in the base folder in S3
def find_csv_files(base_path: str) -> list:
    csv_files = []
    folders_to_scan = [base_path]

    while folders_to_scan:
        current_path = folders_to_scan.pop()
        try:
            items = dbutils.fs.ls(current_path)
        except Exception as e:
            logger.error(f"Could not list path: {current_path} | {e}")
            continue

        for item in items:
            if item.isDir():
                folders_to_scan.append(item.path)
            elif item.name.lower().endswith(".csv"):
                csv_files.append(item.path)

    return csv_files

#To get the table name from the csv file name 

def clean_name_from_path(file_path: str) -> str:
    file_name = file_path.rstrip("/").split("/")[-1]
    raw_table_name = re.sub(r"\.csv$", "", file_name, flags=re.IGNORECASE)
    return re.sub(r"[^0-9a-zA-Z]+", "_", raw_table_name).strip("_").lower()

In [0]:
logger.info(f"Scanning for CSV files under: {base_folder_path}")
all_csv_file_paths = find_csv_files(base_folder_path)
logger.info(f"Found {len(all_csv_file_paths)} CSV file(s) total.")

if table_filter:
    csv_file_paths = [p for p in all_csv_file_paths if clean_name_from_path(p) in table_filter]
    logger.info(f"table_filter applied, processing {len(csv_file_paths)} of {len(all_csv_file_paths)} file(s).")
else:
    csv_file_paths = all_csv_file_paths

for p in csv_file_paths:
    print(p)

#### Audit table

Creates the audit log table if it does not exist yet, used to record the outcome of every file processed in every run.

In [0]:
for sql in [
    "CREATE SCHEMA IF NOT EXISTS cdr_bronze.raw_mimic_iv",
    "CREATE SCHEMA IF NOT EXISTS cdr_silver.processed_mimic_iv",
    "CREATE SCHEMA IF NOT EXISTS cdr_silver.audit",
    "CREATE SCHEMA IF NOT EXISTS cdr_bronze.audit",
]:
    spark.sql(sql)
    print("ok:", sql)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {audit_table} (
    run_id             STRING,
    source_file_path   STRING,
    target_table       STRING,
    load_type          STRING,
    status             STRING,
    row_count          BIGINT,
    rows_inserted      BIGINT,
    rows_updated       BIGINT,
    start_time         TIMESTAMP,
    end_time           TIMESTAMP,
    duration_seconds   DOUBLE,
    notes              STRING,
    error_message      STRING
)
USING DELTA
""")

audit_schema = StructType([
    StructField("run_id",           StringType(),    True),
    StructField("source_file_path", StringType(),    True),
    StructField("target_table",     StringType(),    True),
    StructField("load_type",        StringType(),    True),
    StructField("status",           StringType(),    True),
    StructField("row_count",        LongType(),      True),
    StructField("rows_inserted",    LongType(),      True),
    StructField("rows_updated",     LongType(),      True),
    StructField("start_time",       TimestampType(), True),
    StructField("end_time",         TimestampType(), True),
    StructField("duration_seconds", DoubleType(),    True),
    StructField("notes",            StringType(),    True),
    StructField("error_message",    StringType(),    True),
])

logger.info(f"Audit table ready: {audit_table}")

#### Key uniqueness check

Before merging, checks that the configured key columns are actually unique for the file just read.
If they are not, the pipeline logs a warning and falls back to a row hash key instead of merging on a broken key.

In [0]:
def check_key_uniqueness(df, key_cols):
    """Returns (is_unique, duplicate_row_count) for the given key columns on df."""
    total = df.count()
    distinct = df.dropDuplicates(key_cols).count()
    dup_count = total - distinct
    return dup_count == 0, dup_count

#### Data ingestion

Loads each CSV into bronze. A new table gets a full load, an existing table gets an incremental merge on its configured key, and any table with no key or a failed uniqueness check falls back to a row hash key.

In [0]:
logger.info(f"Starting ingestion run: {run_id} | load_type_param={load_type_param}")

audit_rows = []
success_count = 0
failure_count = 0

for file_path in csv_file_paths:
    file_name = file_path.rstrip("/").split("/")[-1]
    raw_table_name = re.sub(r"\.csv$", "", file_name, flags=re.IGNORECASE)
    clean_table_name = re.sub(r"[^0-9a-zA-Z]+", "_", raw_table_name).strip("_").lower()
    target_table = f"{catalog}.{schema}.{clean_table_name}"

    start_time = datetime.now()
    table_exists = spark.catalog.tableExists(target_table)

    if load_type_param == "historical":
        effective_load_type = "HISTORICAL_FULL_LOAD"
    elif load_type_param == "incremental":
        effective_load_type = "INCREMENTAL_MERGE" if table_exists else "INITIAL_LOAD"
    else:  # auto
        effective_load_type = "INITIAL_LOAD" if not table_exists else "INCREMENTAL_MERGE"

    logger.info(f"Reading {file_path} -> {target_table} [{effective_load_type}]")

    rows_inserted = None
    rows_updated = None
    notes = None

    try:
        raw_df = (spark.read
                  .format("csv")
                  .option("header", "true")
                  .option("inferSchema", "true")
                  .load(file_path))

        source_cols = raw_df.columns
        configured_keys = TABLE_CONFIG.get(clean_table_name)

        use_row_hash = not configured_keys

        if configured_keys and effective_load_type == "INCREMENTAL_MERGE":
            is_unique, dup_count = check_key_uniqueness(raw_df, configured_keys)
            if not is_unique:
                use_row_hash = True
                notes = (
                    f"configured key {configured_keys} was not unique "
                    f"({dup_count} duplicate rows), fell back to row hash key"
                )
                logger.warning(f"  {notes} for {target_table}")

        df = (raw_df
              .withColumn("_ingest_timestamp", current_timestamp())
              .withColumn("_source_file", col("_metadata.file_path"))
              .withColumn("_run_id", lit(run_id)))

        if use_row_hash:
            df = df.withColumn(
                "_row_key",
                sha2(concat_ws("||", *[coalesce(col(c).cast("string"), lit("\u0000")) for c in source_cols]), 256)
            )
            merge_keys = ["_row_key"]
        else:
            merge_keys = configured_keys

        # Force the read to happen now, so a bad file fails here instead of later at write time.
        row_count = df.count()

        if effective_load_type in ("HISTORICAL_FULL_LOAD", "INITIAL_LOAD"):
            is_unique, dup_count = check_key_uniqueness(df, merge_keys)
            if not is_unique:
                logger.warning(
                    f"  {dup_count} duplicate-key row(s) on {merge_keys} for {target_table} "
                    f"-- deduplicating before initial write"
                )
                df = df.dropDuplicates(merge_keys)
                row_count = df.count()

            (df.write
             .format("delta")
             .mode("overwrite")
             .option("overwriteSchema", "true")
             .saveAsTable(target_table))

            rows_inserted = row_count
            rows_updated = 0

        else:  # INCREMENTAL_MERGE
            is_unique, dup_count = check_key_uniqueness(df, merge_keys)
            if not is_unique:
                logger.warning(
                    f"  {dup_count} duplicate-key row(s) on {merge_keys} for {target_table} "
                    f"-- deduplicating before merge"
                )
                df = df.dropDuplicates(merge_keys)

            delta_target = DeltaTable.forName(spark, target_table)
            merge_condition = " AND ".join([f"t.{c} <=> s.{c}" for c in merge_keys])

            (delta_target.alias("t")
            .merge(df.alias("s"), merge_condition)
            .withSchemaEvolution()
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())

            merge_metrics = (delta_target.history(1)
                              .select("operationMetrics")
                              .first()["operationMetrics"])
            rows_inserted = int(merge_metrics.get("numTargetRowsInserted", 0))
            rows_updated = int(merge_metrics.get("numTargetRowsUpdated", 0))

        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()

        logger.info(
            f"  SUCCESS | {row_count} rows read | +{rows_inserted} inserted / "
            f"~{rows_updated} updated | {duration:.1f}s -> {target_table}"
        )

        audit_rows.append(Row(
            run_id=run_id,
            source_file_path=file_path,
            target_table=target_table,
            load_type=effective_load_type,
            status="SUCCESS",
            row_count=row_count,
            rows_inserted=rows_inserted,
            rows_updated=rows_updated,
            start_time=start_time,
            end_time=end_time,
            duration_seconds=duration, 
            notes=notes,
            error_message=None
        ))
        success_count += 1

    except Exception as e:
        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()
        error_message = str(e)[:2000]

        logger.error(f"  FAILED | {file_path} -> {target_table} | {error_message}")

        audit_rows.append(Row(
            run_id=run_id,
            source_file_path=file_path,
            target_table=target_table,
            load_type=effective_load_type,
            status="FAILED",
            row_count=None,
            rows_inserted=None,
            rows_updated=None,
            start_time=start_time,
            end_time=end_time,
            duration_seconds=duration,
            notes=notes,
            error_message=error_message
        ))
        failure_count += 1 
        

#### Update the audit table

Writes all audit rows from this run to the audit table and logs a final summary.

In [0]:
if audit_rows:
    audit_df = spark.createDataFrame(audit_rows, schema=audit_schema)
    audit_df.write.format("delta").mode("append").saveAsTable(audit_table)
    logger.info(f"Audit log written: {len(audit_rows)} row(s) -> {audit_table}")

logger.info(
    f"Run {run_id} complete | {success_count} succeeded | {failure_count} failed "
    f"| {len(csv_file_paths)} total files"
)

if failure_count > 0:
    logger.warning(f"{failure_count} file(s) failed, check {audit_table} for details (run_id = {run_id})")